# 03 · EDA — visualise the dataset
Quick sanity checks before training: sample images, target-length distribution, and table-shape statistics. Bad data is the #1 cause of hallucination, so look first.

In [ ]:
# --- Bootstrap: make the package importable without installing, and stay OFFLINE.
import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
from gemma_ft_json.config import load_config
from gemma_ft_json.tokenization import build_tokenizer

cfg = load_config(ROOT / 'configs' / 'default.yaml')
man = ROOT / 'data' / 'demo' / 'manifests' / 'manifest.jsonl'
assert man.is_file(), 'Run notebook 01 first.'
records = [json.loads(l) for l in open(man).read().splitlines() if l.strip()]
print('records:', len(records))

### Sample a few images with their JSON targets

In [ ]:
n = min(4, len(records))
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1: axes = [axes]
for ax, rec in zip(axes, records[:n]):
    ax.imshow(Image.open(rec['image_path']))
    ax.set_title(rec['target'][:40] + '...', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

### Target token-length distribution (drives `max_target_tokens` and memory)

In [ ]:
tok = build_tokenizer(cfg.model.decoder)
lengths = [len(tok.encode(r['target'], add_bos=False, add_eos=True)) for r in records]
plt.figure(figsize=(6, 4))
plt.hist(lengths, bins=20, color='#4C72B0', edgecolor='white')
plt.xlabel('target length (tokens)'); plt.ylabel('count')
plt.title('Target JSON length distribution'); plt.tight_layout(); plt.show()
print('min/median/max:', min(lengths), sorted(lengths)[len(lengths)//2], max(lengths))

### Table-shape stats (rows per sample), parsed from the JSON targets

In [ ]:
n_rows = []
for r in records:
    try:
        obj = json.loads(r['target'])
        rows = obj.get('rows', obj) if isinstance(obj, dict) else obj
        n_rows.append(len(rows) if isinstance(rows, list) else 1)
    except Exception:
        n_rows.append(0)
plt.figure(figsize=(6, 4))
plt.hist(n_rows, bins=range(0, max(n_rows) + 2), color='#55A868', edgecolor='white', align='left')
plt.xlabel('rows per table'); plt.ylabel('count'); plt.title('Rows-per-table'); plt.tight_layout(); plt.show()